In [1]:
#202431055_Muhammad Rafi Kurniawan
!pip install requests beautifulsoup4 pandas

In [2]:
#202431055_Muhammad Rafi Kurniawan
import re
import csv
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser

In [3]:
#202431055_Muhammad Rafi Kurniawan
BASE_URL   = "https://en.wikipedia.org"
ROBOTS_URL = BASE_URL + "/robots.txt"
HEADERS    = {"User-Agent": "Mozilla/5.0 (compatible; PraktikumScraper/1.0; tugas kuliah web scraping)"}

r = requests.get(ROBOTS_URL, headers=HEADERS, timeout=30)
print("Alamat robots.txt :", ROBOTS_URL)
print("Status code       :", r.status_code)
print("Jumlah baris      :", len(r.text.splitlines()))
print("\n=== 40 baris pertama ===")
print("\n".join(r.text.splitlines()[:40]))

print("\n=== Blok 'User-agent: *' ===")
in_block = False
for ln in r.text.splitlines():
    s = ln.strip()
    if s.lower().startswith("user-agent:"):
        in_block = (s.split(":", 1)[1].strip() == "*")
        if in_block:
            print(s)
        continue
    if in_block and s.lower().startswith(("disallow", "allow", "crawl-delay")):
        print(s)

Alamat robots.txt : https://en.wikipedia.org/robots.txt
Status code       : 200
Jumlah baris      : 712

=== 40 baris pertama ===
﻿# robots.txt for http://www.wikipedia.org/ and friends
#
# Please note: There are a lot of pages on this site, and there are
# some misbehaved spiders out there that go _way_ too fast. If you're
# irresponsible, your access to the site may be blocked.
#

# Observed spamming large amounts of https://en.wikipedia.org/?curid=NNNNNN
# and ignoring 429 ratelimit responses, claims to respect robots:
# http://mj12bot.com/
User-agent: MJ12bot
Disallow: /

# advertising-related bots:
User-agent: Mediapartners-Google*
Disallow: /

# Wikipedia work bots:
User-agent: IsraBot
Disallow:

User-agent: Orthogaffe
Disallow:

# Crawlers that are kind enough to obey, but which we'd rather not have
# unless they're feeding search engines.
User-agent: UbiCrawler
Disallow: /

User-agent: DOC
Disallow: /

User-agent: Zao
Disallow: /

# Some bots are known to be trouble, particular

In [4]:
#202431055_Muhammad Rafi Kurniawan
TARGET_URL = BASE_URL + "/wiki/List_of_countries_and_dependencies_by_population_(United_Nations)"

rp = RobotFileParser()
rp.parse(r.text.splitlines())
print("URL yang akan di-scrape :", TARGET_URL)
print("Dapat diambil (User-agent *)?", rp.can_fetch("*", TARGET_URL))
print("Dapat diambil (User-agent kita)?", rp.can_fetch(HEADERS["User-Agent"], TARGET_URL))

URL yang akan di-scrape : https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_population_(United_Nations)
Dapat diambil (User-agent *)? True
Dapat diambil (User-agent kita)? True


In [5]:
#202431055_Muhammad Rafi Kurniawan
URLS = [TARGET_URL]

pages = []
for u in URLS:
    resp = requests.get(u, headers=HEADERS, timeout=30)
    print(u, "-> status_code:", resp.status_code)
    assert resp.status_code == 200, f"Gagal mengambil {u}"
    pages.append(resp.text)

https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_population_(United_Nations) -> status_code: 200


In [6]:
#202431055_Muhammad Rafi Kurniawan
soups = [BeautifulSoup(html, "html.parser") for html in pages]

def find_data_table(soup):
    """Pilih tabel data utama: wikitable yang header-nya memuat 'continental'."""
    for t in soup.find_all("table", class_="wikitable"):
        if "continental" in t.get_text(" ", strip=True).lower()[:400]:
            return t
    return None

blocks = []
for s in soups:
    table = find_data_table(s)
    assert table is not None, "Tabel data tidak ditemukan (struktur halaman mungkin berubah)"
    blocks.extend(table.find_all("tr"))

print("Jumlah blok (<tr>) ditemukan:", len(blocks))

Jumlah blok (<tr>) ditemukan: 240


In [7]:
#202431055_Muhammad Rafi Kurniawan
def clean(text):
    """Hapus penanda catatan kaki seperti [a], [12]."""
    return re.sub(r"\[[^\]]*\]", "", text).strip()

result = []
for row in blocks:
    for sup in row.find_all("sup"):
        sup.decompose()

    cols = row.find_all(["th", "td"])
    if len(cols) < 6 or row.find("td") is None:
        continue

    negara = clean(cols[0].get_text(strip=True))
    if not negara or negara.lower().startswith("world"):
        continue

    result.append({
        "negara":           negara,
        "populasi_2022":    clean(cols[1].get_text(strip=True)),
        "populasi_2023":    clean(cols[2].get_text(strip=True)),
        "perubahan_persen": clean(cols[3].get_text(strip=True)),
        "benua":            clean(cols[4].get_text(strip=True)),
        "subregion":        clean(cols[5].get_text(strip=True)),
    })

print("Jumlah data hasil scraping:", len(result))
if len(result) < 200:
    print("PERINGATAN: kurang dari 200! Tambahkan halaman lain ke daftar URLS.")

Jumlah data hasil scraping: 237


In [8]:
#202431055_Muhammad Rafi Kurniawan
for i, item in enumerate(result, 1):
    print(i, item)

1 {'negara': 'India', 'populasi_2022': '1,425,423,212', 'populasi_2023': '1,438,069,596', 'perubahan_persen': '+0.89%', 'benua': 'Asia', 'subregion': 'Southern Asia'}
2 {'negara': 'China', 'populasi_2022': '1,425,179,569', 'populasi_2023': '1,422,584,933', 'perubahan_persen': '−0.18%', 'benua': 'Asia', 'subregion': 'Eastern Asia'}
3 {'negara': 'United States', 'populasi_2022': '341,534,046', 'populasi_2023': '343,477,335', 'perubahan_persen': '+0.57%', 'benua': 'Americas', 'subregion': 'Northern America'}
4 {'negara': 'Indonesia', 'populasi_2022': '278,830,529', 'populasi_2023': '281,190,067', 'perubahan_persen': '+0.85%', 'benua': 'Asia', 'subregion': 'South-eastern Asia'}
5 {'negara': 'Pakistan', 'populasi_2022': '243,700,667', 'populasi_2023': '247,504,495', 'perubahan_persen': '+1.56%', 'benua': 'Asia', 'subregion': 'Southern Asia'}
6 {'negara': 'Nigeria', 'populasi_2022': '223,150,896', 'populasi_2023': '227,882,945', 'perubahan_persen': '+2.12%', 'benua': 'Africa', 'subregion': '

In [9]:
#202431055_Muhammad Rafi Kurniawan
CSV_RAW = "populasi_un_raw.csv"
fieldnames = ["negara", "populasi_2022", "populasi_2023", "perubahan_persen", "benua", "subregion"]

with open(CSV_RAW, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for item in result:
        writer.writerow(item)

print(f"Tersimpan: {CSV_RAW} ({len(result)} baris)")

Tersimpan: populasi_un_raw.csv (237 baris)


In [10]:
#202431055_Muhammad Rafi Kurniawan
df = pd.read_csv(CSV_RAW)
print(df.shape)
display(df.head(10))
print("\nTipe data awal:")
print(df.dtypes)

(237, 6)


,negara,populasi_2022,populasi_2023,perubahan_persen,benua,subregion
0,India,"1,425,423,212","1,438,069,596",+0.89%,Asia,Southern Asia
1,China,"1,425,179,569","1,422,584,933",−0.18%,Asia,Eastern Asia
2,United States,"341,534,046","343,477,335",+0.57%,Americas,Northern America
3,Indonesia,"278,830,529","281,190,067",+0.85%,Asia,South-eastern Asia
4,Pakistan,"243,700,667","247,504,495",+1.56%,Asia,Southern Asia
5,Nigeria,"223,150,896","227,882,945",+2.12%,Africa,Western Africa
6,Brazil,"210,306,415","211,140,729",+0.40%,Americas,South America
7,Bangladesh,"169,384,897","171,466,990",+1.23%,Asia,Southern Asia
8,Russia,"145,579,899","145,440,500",−0.10%,Europe,Eastern Europe
9,Mexico,"128,613,117","129,739,759",+0.88%,Americas,Central America



Tipe data awal:
negara              object
populasi_2022       object
populasi_2023       object
perubahan_persen    object
benua               object
subregion           object
dtype: object


In [11]:
#202431055_Muhammad Rafi Kurniawan
def to_number(x):
    """'1,425,423,212' -> 1425423212.0 ; '+0.89%' -> 0.89 ; '−0.2%' -> -0.2 ; gagal -> NaN"""
    s = str(x).replace(",", "").replace("\u2212", "-").replace("%", "").strip()
    m = re.search(r"[-+]?\d+(?:\.\d+)?", s)
    return float(m.group()) if m else float("nan")

num_cols = ["populasi_2022", "populasi_2023", "perubahan_persen"]
for col in num_cols:
    if not pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].apply(to_number)
    df[col] = pd.to_numeric(df[col], errors="coerce")

before = len(df)
df = df.dropna(subset=num_cols).reset_index(drop=True)
print(f"Baris dibuang karena angka tidak valid: {before - len(df)} | sisa: {len(df)}")

print("\nTipe data setelah konversi:")
print(df.dtypes)

CSV_FINAL = "populasi_un_clean.csv"
df.to_csv(CSV_FINAL, index=False)
print(f"\nTersimpan: {CSV_FINAL}")

print("\nJumlah negara/wilayah per benua:")
print(df["benua"].value_counts())
print("\nRata-rata perubahan populasi (%) per benua:")
print(df.groupby("benua")["perubahan_persen"].mean().round(2))
print("\nKorelasi antar atribut angka:")
display(df[num_cols].corr())

Baris dibuang karena angka tidak valid: 0 | sisa: 237

Tipe data setelah konversi:
negara               object
populasi_2022       float64
populasi_2023       float64
perubahan_persen    float64
benua                object
subregion            object
dtype: object

Tersimpan: populasi_un_clean.csv

Jumlah negara/wilayah per benua:
Africa      58
Americas    55
Asia        51
Europe      50
Oceania     23
Name: benua, dtype: int64

Rata-rata perubahan populasi (%) per benua:
benua
Africa      2.13
Americas    0.49
Asia        1.42
Europe      0.21
Oceania     0.22
Name: perubahan_persen, dtype: float64

Korelasi antar atribut angka:


,populasi_2022,populasi_2023,perubahan_persen
populasi_2022,1.000000,0.999979,-0.022720
populasi_2023,0.999979,1.000000,-0.020606
perubahan_persen,-0.022720,-0.020606,1.000000
